# RobustWorld: LTX-Video 2B distilled on Colab

Predicts the second half (target) of each eval clip from its first half (context), then builds a
comparison grid against the ground truth. The sample is 5 clips committed in `data/eval/sample5`.

**Runtime:** GPU. A free **T4** is enough. The distilled model uses 8 steps with no guidance, so
generation takes seconds per clip; most of the time goes on the first download (about 16 GB).

**Secrets** (key icon in the left sidebar, then enable notebook access):
- `GITHUB_TOKEN`: a fine-grained GitHub token with read access to `FinPThomas/RobustWorld` (the repo is private).

No Hugging Face token is needed.

Predictions are saved to Google Drive under `MyDrive/RobustWorld/predictions/`. The other models'
notebooks write to the same place, so the grid at the end includes every model that has been run.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os, subprocess
from google.colab import userdata

REPO = "/content/RobustWorld"
if not os.path.exists(REPO):
    url = f"https://{userdata.get('GITHUB_TOKEN')}@github.com/FinPThomas/RobustWorld.git"
    subprocess.run(["git", "clone", "--depth", "1", url, REPO], check=True, capture_output=True)
    subprocess.run(["git", "-C", REPO, "remote", "set-url", "origin",
                    "https://github.com/FinPThomas/RobustWorld.git"], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=False)
%cd {REPO}
!git log --oneline -1

In [ ]:
!pip install -q -e . -r models/ltx_video/requirements.txt

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
PRED_ROOT = "/content/drive/MyDrive/RobustWorld/predictions"
GRID_ROOT = "/content/drive/MyDrive/RobustWorld/grids"
SAMPLE = "data/eval/sample5"
os.makedirs(PRED_ROOT, exist_ok=True)

## Run LTX-Video

Gets the last 9 context frames (LTX needs 8k+1) and generates the next 16. Each clip's time is printed.

In [ ]:
!python models/ltx_video/predict.py --sample {SAMPLE} --out {PRED_ROOT}/sample5/ltx_video

## Comparison grid

Columns: ground truth, then every model with predictions in Drive. Rows: the input each source
was given (frames a model didn't see are dimmed), then its output. The context plays first,
then the predictions.

In [ ]:
import pathlib
order = ["ltx_video", "cosmos_predict2", "wan21"]            # column order; add new models here
have = [m for m in order if (pathlib.Path(PRED_ROOT) / "sample5" / m).exists()]
out = f"{GRID_ROOT}/sample5/{'+'.join(have)}"
!robustworld-eval grid --sample {SAMPLE} --pred-root {PRED_ROOT} --models {" ".join(have)} --out {out}

In [ ]:
from IPython.display import Video, display
display(Video(f"{out}/all.mp4", embed=True, width=900))